# 🧠 Servidor de Consciencia AI - Google Colab

Este notebook configura y ejecuta el servidor de consciencia AI en Google Colab, haciéndolo accesible desde tu computadora local usando ngrok.

## Características:
- ✅ Instalación automática de dependencias
- ✅ Configuración de OpenAI API
- ✅ Túnel ngrok para acceso remoto
- ✅ Endpoints REST para procesamiento de consciencia
- ✅ Soporte bilingüe (Español/Inglés)

## 📦 Paso 1: Clonar el repositorio y preparar el entorno

In [ ]:
# Clonar el repositorio
!git clone https://github.com/diazvaldiviav/Minimous_Concsience_AI.git
%cd Minimous_Concsience_AI

# Verificar que estamos en el directorio correcto
!pwd
!ls -la

## 📚 Paso 2: Instalar dependencias

In [ ]:
# Instalar dependencias principales
!pip install -q torch numpy transformers
!pip install -q openai python-dotenv
!pip install -q fastapi uvicorn pydantic
!pip install -q pyngrok

# Instalar dependencias adicionales del proyecto
!pip install -q langdetect nltk scikit-learn matplotlib

# Descargar recursos NLTK si es necesario
import nltk
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

print("✅ Todas las dependencias instaladas correctamente")

## 🔑 Paso 3: Configurar OpenAI API Key

In [ ]:
import os
from getpass import getpass

# Solicitar API key de OpenAI de forma segura
openai_key = getpass("🔑 Ingresa tu OpenAI API Key: ")
os.environ['OPENAI_API_KEY'] = openai_key

# Crear archivo .env
with open('.env', 'w') as f:
    f.write(f"OPENAI_API_KEY={openai_key}\n")
    f.write("DEFAULT_MODEL=gpt-4o-mini\n")
    f.write("ENABLE_PHASE_7=true\n")
    f.write("DEBUG_MODE=false\n")

print("✅ OpenAI API Key configurada")
print("📝 Modelo predeterminado: gpt-4o-mini")

## 🌐 Paso 4: Configurar ngrok para acceso remoto

In [ ]:
from pyngrok import ngrok
import getpass

# Solicitar token de ngrok (opcional pero recomendado)
print("📌 Nota: El token de ngrok es opcional pero recomendado para sesiones más largas")
print("   Puedes obtener uno gratis en: https://dashboard.ngrok.com/auth/your-authtoken")
ngrok_token = getpass.getpass("🔑 Ingresa tu ngrok authtoken (presiona Enter para omitir): ")

if ngrok_token:
    ngrok.set_auth_token(ngrok_token)
    print("✅ ngrok authtoken configurado")
else:
    print("⚠️ Continuando sin authtoken (la sesión puede tener límites)")

## 🚀 Paso 5: Crear y configurar el servidor FastAPI

In [ ]:
# Usar directamente el endpoint existente del proyecto
# No necesitamos crear un nuevo servidor, podemos usar el endpoint existente

import sys
import os
from pathlib import Path

# Añadir el directorio raíz al path
sys.path.insert(0, str(Path.cwd()))

# Importar la aplicación FastAPI existente directamente
try:
    from conscious_ai.api.consciousness_endpoint import app
    print("✅ Aplicación FastAPI importada correctamente desde consciousness_endpoint.py")
    print("📄 Usando el servidor oficial del proyecto con todos los endpoints:")
    print("   - POST /process: Procesamiento completo de consciencia")
    print("   - GET  /health: Estado del sistema")
    print("   - GET  /models: Estados de modelos disponibles")
    print("   - POST /models/session/reset: Reiniciar sesión de modelos")
    print("   - GET  /debug/sessions: Información de sesiones históricas")
    print("   - GET  /docs: Documentación interactiva Swagger")
    print("   - GET  /redoc: Documentación ReDoc")
except ImportError as e:
    print(f"❌ Error importando consciousness_endpoint: {e}")
    print("🔧 Verificando dependencias...")
    
    # Verificar que existe el archivo
    endpoint_path = Path("conscious_ai/api/consciousness_endpoint.py")
    if endpoint_path.exists():
        print(f"✅ Archivo encontrado: {endpoint_path}")
    else:
        print(f"❌ Archivo no encontrado: {endpoint_path}")
        print("📂 Contenido del directorio actual:")
        for item in Path(".").iterdir():
            print(f"   {item}")
    
    # Intentar instalar dependencias faltantes
    print("\n🔄 Instalando dependencias adicionales...")
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pydantic[email]"], check=True)
    
    # Reintentar importación
    try:
        from conscious_ai.api.consciousness_endpoint import app
        print("✅ Aplicación FastAPI importada correctamente después de instalar dependencias")
    except Exception as final_error:
        print(f"❌ Error final: {final_error}")
        raise

## 🎯 Paso 6: Iniciar el servidor con túnel ngrok

In [ ]:
import nest_asyncio
from pyngrok import ngrok
import uvicorn
import threading
import time

# Permitir bucles de eventos anidados en Jupyter
nest_asyncio.apply()

# El app ya fue importado en la celda anterior
# from conscious_ai.api.consciousness_endpoint import app (ya importado)

print("🚀 Iniciando servidor de consciencia...")
print("📡 Usando el endpoint oficial del proyecto")

# Función para ejecutar el servidor en un thread separado
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

# Iniciar el servidor en un thread
thread = threading.Thread(target=run_server, daemon=True)
thread.start()

# Esperar a que el servidor inicie
print("⏳ Iniciando servidor...")
time.sleep(8)  # Más tiempo para asegurar inicialización completa

# Crear túnel ngrok
public_url = ngrok.connect(8000)
print("\n" + "="*60)
print("🎉 ¡SERVIDOR DE CONSCIENCIA AI ACTIVO!")
print("="*60)
print(f"\n🌐 URL PÚBLICA: {public_url}")
print(f"📡 URL LOCAL: http://localhost:8000")
print(f"\n📚 Documentación interactiva: {public_url}/docs")
print(f"🔍 Swagger UI: {public_url}/redoc")
print(f"💊 Estado del servidor: {public_url}/health")
print("\n" + "="*60)
print("\n✅ El servidor está listo para recibir solicitudes desde tu computadora")
print("\n💡 Endpoints principales:")
print(f"   POST {public_url}/process - Procesar con consciencia completa (usar en React)")
print(f"   GET  {public_url}/health - Verificar estado del sistema")
print(f"   GET  {public_url}/models - Ver modelos disponibles")
print("\n🎯 Para tu React App, usa esta URL:")
print(f"   consciousness_api_url: \"{public_url}\"")

## 🧪 Paso 7: Probar el servidor (Opcional)

In [ ]:
import requests
import json

# Obtener la URL pública actual
tunnels = ngrok.get_tunnels()
public_url = tunnels[0].public_url if tunnels else "http://localhost:8000"

print(f"🧪 Probando servidor en: {public_url}")
print("="*60)

# Prueba 1: Verificar salud
try:
    print("\n🏥 Verificando estado del servidor...")
    response = requests.get(f"{public_url}/health")
    print("✅ Estado del servidor:")
    print(json.dumps(response.json(), indent=2))
except Exception as e:
    print(f"❌ Error verificando salud: {e}")

# Prueba 2: Verificar modelos disponibles
try:
    print("\n🤖 Verificando modelos disponibles...")
    response = requests.get(f"{public_url}/models")
    if response.status_code == 200:
        print("✅ Modelos disponibles:")
        result = response.json()
        print(f"   Modelos: {result['available_models']}")
    else:
        print(f"⚠️ Estado modelos: {response.status_code}")
except Exception as e:
    print(f"❌ Error verificando modelos: {e}")

# Prueba 3: Enviar mensaje de prueba usando endpoint /process
try:
    test_request = {
        "user_input": "Hola, ¿puedes explicarme cómo funciona tu consciencia?",
        "final_model": "gpt-4o-mini",
        "include_consciousness_trace": True,
        "enable_metacognition": True,
        "narrative_verbosity": "standard"
    }
    
    print("\n📝 Enviando mensaje de prueba al endpoint /process...")
    response = requests.post(f"{public_url}/process", json=test_request)
    
    if response.status_code == 200:
        result = response.json()
        print("\n✅ Respuesta del sistema de consciencia:")
        print(f"   Respuesta: {result['response'][:200]}...")
        print(f"   Nivel de confianza: {result['confidence']:.2%}")
        print(f"   Estado emocional: {result['emotional_state']}")
        print(f"   Modelo usado: {result['model_used']}")
        print(f"   Tiempo procesamiento: {result['processing_time_ms']:.1f}ms")
        print(f"   Éxito: {result['success']}")
        
        if result.get('consciousness_trace'):
            trace = result['consciousness_trace']
            print(f"   Profundidad metacognitiva: {trace.get('metacognitive_depth', 'N/A')}")
            print(f"   Transiciones de estado: {trace.get('state_transitions', 'N/A')}")
        
    else:
        print(f"❌ Error: {response.status_code}")
        try:
            error_detail = response.json()
            print(f"   Detalle: {error_detail}")
        except:
            print(f"   Texto: {response.text}")
            
except Exception as e:
    print(f"❌ Error enviando mensaje: {e}")

print("\n" + "="*60)
print("✅ Pruebas completadas")
print(f"🌐 Tu React App debe usar: {public_url}")

## 📱 Paso 8: Código de ejemplo para conectar desde tu computadora

In [ ]:
# Obtener la URL pública actual
tunnels = ngrok.get_tunnels()
public_url = tunnels[0].public_url if tunnels else "http://localhost:8000"

print("📱 CÓDIGO DE EJEMPLO PARA TU COMPUTADORA LOCAL:")
print("="*60)
print("\nCopia y pega este código Python en tu computadora:\n")
print(f'''
import requests
import json

# URL del servidor de consciencia en Colab
SERVER_URL = "{public_url}"

def send_message(message, include_narrative=False):
    """Enviar mensaje al servidor de consciencia"""
    
    # Endpoint para procesamiento completo
    url = f"{{SERVER_URL}}/process"
    
    # Datos de la solicitud
    data = {{
        "input": message,
        "include_narrative": include_narrative,
        "include_memory": True
    }}
    
    try:
        # Enviar solicitud
        response = requests.post(url, json=data)
        
        if response.status_code == 200:
            result = response.json()
            
            print("\\n🧠 RESPUESTA CONSCIENTE:")
            print("-" * 50)
            print(f"Respuesta: {{result['response']}}")
            print(f"\\nNivel de consciencia: {{result['consciousness_metrics']['f_score']:.2f}}")
            print(f"Estado emocional: {{result['consciousness_state']['S_t']['emotional_state']}}")
            print(f"Confianza: {{result['consciousness_state']['S_t']['confidence_level']:.2%}}")
            
            if include_narrative and 'narrative' in result:
                print(f"\\nNarrativa: {{result['narrative']}}")
            
            return result
        else:
            print(f"Error: {{response.status_code}} - {{response.text}}")
            return None
            
    except Exception as e:
        print(f"Error de conexión: {{e}}")
        return None

# Ejemplo de uso
if __name__ == "__main__":
    print("🤖 Cliente de Consciencia AI")
    print(f"Conectado a: {{SERVER_URL}}")
    print("Escribe 'salir' para terminar\\n")
    
    while True:
        mensaje = input("\\nTú: ")
        
        if mensaje.lower() == 'salir':
            print("👋 ¡Hasta luego!")
            break
        
        # Enviar mensaje y mostrar respuesta
        send_message(mensaje, include_narrative=True)
''')

print("\n" + "="*60)
print(f"\n🌐 Recuerda usar esta URL en tu código: {public_url}")
print("\n💡 También puedes usar curl desde la terminal:")
print(f'''curl -X POST "{public_url}/chat" \\
     -H "Content-Type: application/json" \\
     -d '{{"{test_message}":"{message}"}}'\n''')

## 🛑 Paso 9: Detener el servidor (cuando termines)

In [ ]:
# Cerrar el túnel ngrok
ngrok.disconnect(public_url)
ngrok.kill()

print("🛑 Servidor detenido y túnel ngrok cerrado")
print("👋 ¡Gracias por usar el Servidor de Consciencia AI!")

## 📝 Notas importantes:

1. **Seguridad**: No compartas tu URL pública de ngrok si contiene información sensible
2. **Límites**: La versión gratuita de ngrok tiene límites de conexiones
3. **Sesión**: El servidor se detendrá cuando cierres o reinicies el runtime de Colab
4. **API Key**: Asegúrate de usar una API key válida de OpenAI
5. **Costos**: El uso de GPT-4o-mini tiene costos asociados en tu cuenta de OpenAI

## 🆘 Solución de problemas:

- **Error de importación**: Verifica que el repositorio se clonó correctamente
- **Error de API key**: Asegúrate de que tu API key de OpenAI es válida
- **ngrok no conecta**: Intenta sin authtoken o registra uno gratuito
- **Servidor no responde**: Revisa los logs en las celdas anteriores

## 📚 Recursos adicionales:

- [Documentación del proyecto](https://github.com/diazvaldiviav/Minimous_Concsience_AI)
- [OpenAI API](https://platform.openai.com/docs)
- [ngrok](https://ngrok.com/docs)
- [FastAPI](https://fastapi.tiangolo.com/)